# Train short — "The 4-Cent Part That Grounded High-Speed Trains" (Blender, Eevee on Colab GPU)

122-word script, Warm Pebble voice (50.184 s), 15 shots, 2D flat human characters
(@RicksClips3000 style) in own-built 3D sets — every phrase literally enacted.
Built with scripted geometry (bpy), rendered here with Eevee on a Colab T4 GPU.

**What this notebook does, in order:**
1. Mounts Drive + checks a GPU is attached (fails fast if not).
2. Installs Blender 4.2.23 + the system GL libs it needs.
3. Clones the project repo (`attempss7-tech/blender-train-short`).
4. Renders **15 Eevee test frames (one per shot)** — **STOP and inspect these**
   before the full render (2D characters upright/flat/facing, gauge needles and
   clock hands visible, S02 map band light blue not black).
5. Renders all 1,506 frames in **3 shards**, saving each batch to Drive
   immediately (a dropped session loses at most one batch, ~5 min).
6. Assembles the final mp4 (fades + sound mix) and verifies it.

**Before you run:** Runtime > Change runtime type > **T4 GPU** > Save.
Full click-path + what to do if a session drops: see `COLAB_RUN.md` in the repo.


In [ ]:
# Setup: mount Drive, check the GPU, define paths + a streaming run() helper.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, shutil, glob, json, time

REPO_DIR = '/content/blender-train-short'
BLENDER = '/content/blender-4.2.23-linux-x64/blender'
DRIVE_ROOT = '/content/drive/MyDrive/train_blender'
DRIVE_FRAMES = DRIVE_ROOT + '/frames'
DRIVE_TEST = DRIVE_ROOT + '/test_frames'
DRIVE_FINAL = DRIVE_ROOT + '/final'
LOCAL_FRAMES = '/content/frames'

def run(cmd, check=True):
    print('+', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed ({p.returncode}): {" ".join(cmd)}')
    return ''.join(out)

# GPU check — fail fast with a clear message if there is no GPU.
try:
    smi = run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
    print('GPU OK:', smi.strip())
except Exception:
    raise SystemExit('NO GPU ATTACHED — Runtime > Change runtime type > T4 GPU, then re-run this cell.')
for d in (DRIVE_FRAMES, DRIVE_TEST, DRIVE_FINAL, LOCAL_FRAMES):
    os.makedirs(d, exist_ok=True)
print('Paths ready.')


In [ ]:
# Install system GL libs + Blender 4.2.23 (same version the project was built with).
# Takes ~2-4 min. Re-running is safe (skips the download if already extracted).
run(['apt-get', 'update', '-qq'])
run(['env', 'DEBIAN_FRONTEND=noninteractive', 'apt-get', 'install', '-y', '-qq',
     'libegl1', 'libgl1', 'libegl-mesa0'])
if not os.path.exists(BLENDER):
    run(['curl', '-sSL', '-o', '/content/blender.tar.xz',
         'https://download.blender.org/release/Blender4.2/blender-4.2.23-linux-x64.tar.xz'])
    run(['tar', '-xf', '/content/blender.tar.xz', '-C', '/content'])
    os.remove('/content/blender.tar.xz')
run([BLENDER, '--version'])
print('Blender ready:', BLENDER)

# EGL FIX (required on Colab, proven 2026-11-12): Colab ships the NVIDIA EGL
# library at /usr/lib64-nvidia/libEGL_nvidia.so.0 but NO vendor file for it, so
# Blender's EGL context fails (EGL_BAD_MATCH) and Eevee silently falls back to
# CPU/llvmpipe at ~5 min/frame. Registering the vendor file makes Eevee use the
# GPU (~3.5 s/frame on a T4). The file lives only in this session's filesystem,
# so this cell must run again after any session drop (recovery re-runs setup).
import glob as _glob
_nvidia_egl = (_glob.glob('/usr/lib64-nvidia/libEGL_nvidia.so.0')
               + _glob.glob('/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0'))
if _nvidia_egl:
    os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
    with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as _f:
        json.dump({"file_format_version": "1.0.0",
                   "ICD": {"library_path": "libEGL_nvidia.so.0"}}, _f, indent=2)
    print('EGL vendor file written (NVIDIA EGL at', _nvidia_egl[0] + ')')
else:
    print('WARNING: libEGL_nvidia.so.0 not found — Eevee may fall back to CPU.')
    print('Check the test-frames timings: minutes/frame means CPU fallback.')


In [ ]:
# Clone the project repo (or update it if already cloned).
if os.path.exists(REPO_DIR + '/.git'):
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    run(['git', 'clone', 'https://github.com/attempss7-tech/blender-train-short.git', REPO_DIR])
run(['ls', '-la', REPO_DIR + '/scenes/', REPO_DIR + '/assets/'])


In [ ]:
# GPU probe + Eevee TEST FRAMES (one per shot — 15 frames).
# This is the mandatory look check: Eevee was never validated locally (no GPU
# there). Inspect the 15 frames below BEFORE running any shard cell.
run(['python3', f"{REPO_DIR}/scripts/colab_render.py",
     '--blender', BLENDER, '--gpu-probe'])
run(['python3', f"{REPO_DIR}/scripts/colab_render.py",
     '--blender', BLENDER, '--blend', f"{REPO_DIR}/scenes/train.blend",
     '--out', '/content/test_frames', '--frames', '40,130,245,360,480,575,645,720,800,900,1020,1130,1240,1350,1460'])
for f in glob.glob('/content/test_frames/*.png'):
    shutil.copy2(f, DRIVE_TEST + '/' + os.path.basename(f))
print('Test frames saved to Drive:', DRIVE_TEST)

from IPython.display import Image, display
for fr in [40, 130, 245, 360, 480, 575, 645, 720, 800, 900, 1020, 1130, 1240, 1350, 1460]:
    p = f'/content/test_frames/f{fr:04d}.png'
    print(f'--- frame {fr} ---')
    display(Image(p, width=270))


## ⏸ STOP — inspect the 15 test frames above before continuing

Check each frame for:
- **2D characters upright, flat, facing camera** (f575 passengers, f645 technicians, f800 technician, f1130 technician) — no black shells, no missing limbs.
- **f130 (S02 map): the top of the frame must be LIGHT BLUE** (backdrop), not black — a black band here means an Eevee lighting problem, stop and report it.
- **Gauge needle + red zone visible** (f1130 rest, f1155 spiking — re-render f1155 with the cell below if unsure), **clock hands + delay dial visible** (f1240).
- **Red alert lamp bright** (f290), **sparks** (f525), **bubbles** (f1020), **crack glow** (f890).
- No black/missing objects anywhere; washer alone on its pedestal (f40); exploded wheel parts spread (f670).

**If anything looks wrong:** stop, report the frame number + what you see, and do NOT run the shards.
**If all 15 frames look right:** skip the re-render cell and run the 3 shard cells.


In [ ]:
# RE-RENDER ONE FRAME — only run if a test frame needs a closer look
# (e.g. f1155 for the gauge spike, f290 for the red alert, f670 for the full
# explosion). Edit FRAME below, run, and inspect the image.
FRAME = 1155  # <-- edit me
run(['python3', f'{REPO_DIR}/scripts/colab_render.py',
     '--blender', BLENDER, '--blend', f'{REPO_DIR}/scenes/train.blend',
     '--out', '/content/test_frames', '--frames', str(FRAME)])
from IPython.display import Image, display
display(Image(f'/content/test_frames/f{FRAME:04d}.png', width=270))


In [ ]:
# SHARD 1: frames 0-501 (502 frames, batches of 84).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 0, 501, 84
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/train.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 1 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# SHARD 2: frames 502-1003 (502 frames, batches of 84).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 502, 1003, 84
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/train.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 2 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# SHARD 3: frames 1004-1505 (502 frames, batches of 84).
# Resume-safe: syncs Drive -> local first, renders only missing frames,
# copies each batch to Drive immediately, then deletes the local batch.
SHARD_START, SHARD_END, BATCH = 1004, 1505, 84
os.makedirs(LOCAL_FRAMES, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)
# 1. Resume sync: pull any frames already on Drive for this shard.
synced = 0
for f in range(SHARD_START, SHARD_END + 1):
    d = f"{DRIVE_FRAMES}/f{f:04d}.png"
    l = f"{LOCAL_FRAMES}/f{f:04d}.png"
    if os.path.exists(d) and not os.path.exists(l):
        shutil.copy2(d, l); synced += 1
print(f"Resume sync: {synced} frames restored from Drive.")
# 2. Render + copy, one batch at a time.
for bs in range(SHARD_START, SHARD_END + 1, BATCH):
    be = min(bs + BATCH - 1, SHARD_END)
    t0 = time.time()
    run(["python3", f"{REPO_DIR}/scripts/colab_render.py",
         "--blender", BLENDER, "--blend", f"{REPO_DIR}/scenes/train.blend",
         "--out", LOCAL_FRAMES, "--start", str(bs), "--end", str(be)])
    copied, missing = 0, []
    for f in range(bs, be + 1):
        s, d = f"{LOCAL_FRAMES}/f{f:04d}.png", f"{DRIVE_FRAMES}/f{f:04d}.png"
        if os.path.exists(s):
            if not os.path.exists(d):
                shutil.copy2(s, d)
            copied += 1
            os.remove(s)
        else:
            missing.append(f)
    print(f"Batch {bs}-{be}: {copied} frames on Drive, "
          f"{time.time() - t0:.0f}s, missing={missing if missing else 'none'}", flush=True)
    if missing:
        raise RuntimeError(f"Batch {bs}-{be} incomplete, missing {missing} — re-run this cell.")
# 3. Shard verification.
have = sum(1 for f in range(SHARD_START, SHARD_END + 1)
           if os.path.exists(f"{DRIVE_FRAMES}/f{f:04d}.png"))
print(f"SHARD 3 DONE: {have}/{SHARD_END - SHARD_START + 1} frames on Drive.")
assert have == SHARD_END - SHARD_START + 1, "Shard incomplete — re-run this cell."


In [ ]:
# ASSEMBLY: verify all 1,506 frames, then ffmpeg -> final mp4.
# Fades (ffmpeg, not baked into the scene): fade-in f0-14 (0.5s),
# fade-out f1491-1505 (0.5s from 49.684s — INSIDE the voice, no dead silence
# at the end; last word ends 49.95s). Mix is already -14.4 LUFS — mux as-is.
missing = [f for f in range(1506) if not os.path.exists(f'{DRIVE_FRAMES}/f{f:04d}.png')]
print(f'Frames on Drive: {1506 - len(missing)}/1506, missing: {missing[:20]}')
assert not missing, f'{len(missing)} frames missing — re-run the shard cell(s) covering them.'
run(['ffmpeg', '-y',
     '-framerate', '30', '-i', f'{DRIVE_FRAMES}/f%04d.png',
     '-i', f'{REPO_DIR}/assets/mix.wav',
     '-vf', 'fade=t=in:st=0:d=0.5,fade=t=out:st=49.684:d=0.5,format=yuv420p',
     '-c:v', 'libx264', '-crf', '17', '-preset', 'medium',
     '-c:a', 'aac', '-b:a', '192k', '-shortest', '-movflags', '+faststart',
     '/content/train_blender.mp4'])
shutil.copy2('/content/train_blender.mp4', DRIVE_FINAL + '/train_blender.mp4')
print('Final video:', DRIVE_FINAL + '/train_blender.mp4')


In [ ]:
# VERIFICATION: ffprobe + loudness of the final mp4.
run(['ffprobe', '-v', 'error', '-show_entries',
     'format=duration,size:stream=width,height,r_frame_rate,codec_name',
     '-of', 'default=noprint_wrappers=1', DRIVE_FINAL + '/train_blender.mp4'])
run(['ffmpeg', '-i', DRIVE_FINAL + '/train_blender.mp4',
     '-vn', '-af', 'ebur128', '-f', 'null', '-'])
print('Expected: ~50.18s, 1080x1920, 30fps, h264+AAC, ~-14.4 LUFS.')


## Done

The final video is at `MyDrive/train_blender/final/train_blender.mp4`,
with all 1,506 frames kept at `MyDrive/train_blender/frames/` and the Eevee
test frames at `MyDrive/train_blender/test_frames/`.

**Optional — rebuild the .blend from scripts** (only if the committed .blend
ever fails to load/render; takes ~1 min, overwrites `scenes/train.blend`):
```
!cd /content/blender-train-short && /content/blender-4.2.23-linux-x64/blender --background --python scripts/build_all.py
```
